In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
# from statsmodels.stats.multitest import multipletests
# from sklearn.linear_model import LinearRegression
# from sklearn.metrics import r2_score 

: 

In [ ]:
df_prev = pd.read_csv('all_results_old_and_new.csv')
df_prev.shape

In [ ]:
df_babylm_10m = df_prev[df_prev.exp_info.str.startswith('babylm') & df_prev.encoding_setting.str.contains('default') & df_prev.response_setting.str.contains('default')]
df_babylm_10m

In [ ]:
df_llama = df_prev[df_prev.exp_info.str.startswith('LLAMA') & df_prev.encoding_setting.str.contains('default') & df_prev.response_setting.str.contains('default')]
df_llama

In [ ]:
df_curr = pd.read_csv('all_results_1207.csv') #1111
#df_curr

In [ ]:
df_curr_wo_llama = df_curr[~df_curr.exp_info.str.startswith('llama')]
df_curr_wo_llama = df_curr_wo_llama[~df_curr_wo_llama.exp_info.str.startswith('babyllama-10m')]
df_curr_wo_llama

In [ ]:
df_all = pd.concat([df_llama, df_curr_wo_llama, df_babylm_10m], ignore_index=True) #107+23+7 = 138
df_all

In [ ]:
df_all.exp_info.str.startswith('babyllama-100m').sum()

In [ ]:
df_all[df_all.stimulus_setting.str.startswith('babylm-babyllama-10m')] 


In [ ]:
df_all.to_csv('all_results_combined_1207.csv') #1111

In [ ]:
#EXCLUDE pythia-70m for now
#df_all = df_all[~df_all.exp_info.str.startswith('pythia-70m')]
df_all

## PLOT!

In [ ]:

import pandas as pd
import re

# Load the data
#df = pd.read_csv('all_results_old_and_new.csv')

# Filter for default settings
#df_filtered = df[(df['encoding_setting'] == 'default') & (df['response_setting'] == 'default')]

df_filtered=df_all

# Function to extract layer number
def extract_layer(stimulus_setting):
    match = re.search(r'_(\d+)$', stimulus_setting)
    if match:
        return int(match.group(1))
    return None

# Function to extract model base name (without layer number)
def extract_model_base(stimulus_setting):
    # Split by underscore and check if last part is a number
    parts = stimulus_setting.split('_')
    if len(parts) >= 2 and parts[-1].isdigit():
        # Remove the last part (layer number) to get base model name
        base = '_'.join(parts[:-1])
        return base
    return stimulus_setting

# Add columns
df_filtered = df_filtered.copy()
df_filtered['model_base'] = df_filtered['stimulus_setting'].apply(extract_model_base)
df_filtered['layer'] = df_filtered['stimulus_setting'].apply(extract_layer)

# Remove rows where layer is None
df_filtered = df_filtered.dropna(subset=['layer'])

# Create NUM_LAYERS_DICT
NUM_LAYERS_DICT = {}

print("Extracting NUM_LAYERS_DICT from data:")
print("=" * 60)

for model_base in sorted(df_filtered['model_base'].unique()):
    model_data = df_filtered[df_filtered['model_base'] == model_base]
    max_layer = int(model_data['layer'].max())
    all_layers = sorted([int(x) for x in model_data['layer'].unique()])
    
    # The total number of layers is typically max_layer + 1
    # (since layer indexing often starts from 0)
    # But let's also check if there's a pattern
    
    # Check if layers start from 0 or 1
    min_layer = int(model_data['layer'].min())
    
    if min_layer == 0:
        # Layers indexed from 0, so total layers = max_layer + 1
        total_layers = max_layer + 1
    else:
        # Layers might be indexed from 1, or sampled differently
        total_layers = max_layer + 1
    
    NUM_LAYERS_DICT[model_base] = total_layers
    
    print(f"\n{model_base}:")
    print(f"  Layers tested: {all_layers}")
    print(f"  Min layer: {min_layer}, Max layer: {max_layer}")
    print(f"  Inferred total layers: {total_layers}")

NUM_LAYERS_DICT


In [ ]:
max_layer_indices = df_filtered.groupby('model_base')['layer'].max()
NUM_LAYERS_DICT = (max_layer_indices + 1).to_dict()

print("Dynamically calculated total layers (N) for each model:")
print(NUM_LAYERS_DICT)

In [ ]:
from utils_perf_df import analyze_custom_settings_combinations, filter_sort_and_convert_to_numeric
from utils_plotter import plot_baseline_value
import matplotlib.pyplot as plt
import pandas as pd

all_results = df_all

columns_of_interest = [ 'mean_cc_norm_mean', 'mean_cc_norm_std', 'mean_r2_mean', 'mean_r2_std', 'stimulus_setting', 'stimulus_setting_2', 'encoding_setting', 'response_setting', 'format_name_name'] #, 'name_setting', 'task', 'format_name_name'] 'exp_info',

final_results = all_results[columns_of_interest]
final_results = final_results.reset_index(drop = True)
final_results = final_results[final_results['format_name_name'].str.contains('loss_criteria__MAE|default')]

settings_list = ['stimulus_setting', 'encoding_setting', 'response_setting', 'stimulus_setting_2']
results = analyze_custom_settings_combinations(final_results, settings_list)


In [ ]:
results

In [ ]:
# --- Define the Model-to-Color Mapping (Place this near the model2name_dict) ---

model2name_dict = {
    "LLAMA7B": "LLAMA 7B",
    "LLAMA13B": "LLAMA 13B",
    "LLAMA30B": "LLAMA 33B",
    "LLAMA65B": "LLAMA 65B",
    "LLAMA27Bhf": "LLAMA-2 7B",
    "LLAMA38B": "LLAMA-3 8B",
    "babylm-babyllama-10m-2024": "BabyLLaMA 58M 10M",
    "babylm/babyllama-100m-2024": "BabyLLaMA 58M 100M",
    "EleutherAI/pythia-14m": "Pythia 14M",
    "EleutherAI/pythia-70m": "Pythia 70M",
    'babylm/opt-125m-strict-2023': "OPT 125M 100M",
    'babylm/opt-125m-strict-small-2023': "OPT 125M 10M",
    "babylm/ltgbert-100m-2024": "LTGBERT 100M",
    "babylm/ltgbert-10m-2024": "LTGBERT 10M",
    "babylm/t5-base-strict-2023" : "T5 Base Strict 100M",
    'babylm/t5-base-strict-small-2023': "T5 Base Strict Small 10M",
    'EleutherAI/pythia-160m': "Pythia 160M"
    
    
}

MODEL_COLORS = {
    "LLAMA7B": "tab:blue",
    "LLAMA13B": "tab:orange",
    "LLAMA30B": "tab:green",    # For LLAMA 33B
    "LLAMA65B": "tab:red",
    "LLAMA27Bhf": "tab:orange", # For LLAMA-2 7B
    "LLAMA38B": "tab:green",    # For LLAMA-3 8B
    "babylm-babyllama-10m-2024": "tab:purple", 
    "babylm/babyllama-100m-2024": "tab:pink",
    "EleutherAI/pythia-14m": "tab:brown",
    "EleutherAI/pythia-70m": "tab:cyan",
    'babylm/opt-125m-strict-2023': "tab:gray",
    'babylm/opt-125m-strict-small-2023': "tab:olive",
    "babylm/ltgbert-100m-2024": "black",
    "babylm/ltgbert-10m-2024": "navy",
    "babylm/t5-base-strict-2023" : "darkgray",
    "babylm/t5-base-strict-small-2023": "lightgray",
    "EleutherAI/pythia-160m": "brown"   
}
# ---------------------------------------------------------------------------------

# # Define the criteria and x-axis
criteria = 'mean_r2_mean'
x_axis = 'stimulus_setting'


def extra_plot_settings(ax, title, xlim, ylim, criteria, xlabel="Normalized Layer Depth", ylabel="Encoder Performance (Avg $r^2$)", fontsize=16, loc = 'lower right' , include_legend=True, **kwargs):
    ax.set_xlabel(xlabel, fontsize=fontsize)
    ax.set_ylabel(ylabel, fontsize=fontsize)
    ax.set_title(title, fontsize=fontsize)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.tick_params(axis='both', labelsize=fontsize)

    if include_legend:
        ax.legend(fontsize=fontsize, loc=loc)
    plot_baseline_value(criteria, ax_or_plt=ax, **kwargs)
    

# Function to plot data for given models (CORRECTED to use MODEL_COLORS)
def plot_models(ax, models, title, xlabel, ylabel, xlim, ylim, fontsize = 12, **kwargs):
    
    for model in models:
        # 🎯 Determine the base model key (e.g., 'LLAMA7B', not 'LLAMA7B_layer_2')
        model_key = model.split('_')[0]
        
        # 🎨 Get the color from the global map. Default to black if key is missing.
        color_to_use = MODEL_COLORS.get(model_key, 'black')
        
        df = results['encoding_setting__response_setting'][("default", "default")]
        # Use a raw string r'' for the regex pattern
        df = df[df['stimulus_setting'].str.contains(r'^' + model, na=False)]
        numerized_df = filter_sort_and_convert_to_numeric(df, {}, multimodal=False, sort_by=x_axis,
                                                          colunmns2convert=['stimulus_setting'])
        
        ax.plot(
            numerized_df[x_axis] / (NUM_LAYERS_DICT[model_key] - 1), 
            numerized_df[criteria], 
            label=model2name_dict[model_key], 
            marker='+', 
            linewidth=3,
            color=color_to_use # <--- KEY CHANGE: Use the specific model color
        )
    extra_plot_settings(ax, title, xlim, ylim, criteria, xlabel, ylabel, fontsize = fontsize, **kwargs)

# NOTE: You must ensure NUM_LAYERS_DICT uses these base keys: 'LLAMA7B', 'LLAMA13B', etc.
# If it currently uses 'LLAMA7Bhf', update the code to handle that split. 
# Based on the snippet, I've adjusted the code to use model.split('_')[0] for both color lookup and NUM_LAYERS_DICT lookup.

fig, axs = plt.subplots(2, 2, figsize=(10, 7))

# Plot LLAMA7B version variations in the first row

# Plot BabyLM and all models comparison in the second row
plot_models(axs[0, 0], ["babylm-babyllama-10m-2024", "babylm/babyllama-100m-2024", "babylm/opt-125m-strict-2023", "babylm/opt-125m-strict-small-2023", "EleutherAI/pythia-14m", "EleutherAI/pythia-70m",  "EleutherAI/pythia-160m", "babylm/ltgbert-100m-2024", "babylm/ltgbert-10m-2024", "babylm/t5-base-strict-2023", "babylm/t5-base-strict-small-2023"], None, None, None, (0, 1), (0.002, 0.0035))

#plot_models(axs[0, 0], ["LLAMA7B", "LLAMA27Bhf", "LLAMA38B"], None, None, None, (0, 1), (0, 0.04))
plot_models(axs[0, 1], ["LLAMA7B", "LLAMA13B", "LLAMA30B", "LLAMA65B"], None, None, None, (0, 1), (0, 0.04))

plot_models(axs[1, 0], ["EleutherAI/pythia-14m", "EleutherAI/pythia-70m", "EleutherAI/pythia-160m", "babylm/ltgbert-100m-2024", "babylm/ltgbert-10m-2024"], None, None, None, (0, 1), (0, 0.02))

plot_models(axs[1, 1], ["LLAMA7B", "LLAMA13B", "LLAMA30B", "LLAMA65B", "babylm-babyllama-10m-2024", "babylm/babyllama-100m-2024", "babylm/opt-125m-strict-2023", "babylm/opt-125m-strict-small-2023", "EleutherAI/pythia-14m", "EleutherAI/pythia-70m","babylm/ltgbert-100m-2024", "babylm/ltgbert-10m-2024"], None, None, None, (0, 1), (0, 0.04), include_legend=False)


# Plot Whisper version variations in the second row
#plot_models(axs[1, 0], ['whisper-tiny_(\\d+|\\w+)+_default_default', 'whisper-small_(\\d+|\\w+)_default_default', 'whisper-base_(\\d+|\\w+)_default_default','whisper-large_(\\d+|\\w+)+_default_default'], None, None, None, (0, 1), (0, 0.03), whisper=True)
#plot_models(axs[1, 1], ['whisper-large_(\\d+|\\w+)+_default_default', 'whisper-large-v2_(\\d+|\\w+)_default_default', 'whisper-large-v3_(\\d+|\\w+)_default_default'], None, None, None , (0, 1), (0, 0.03), whisper=True)

# Add labels (a, b, c, d) on top of each subplot
labels = ['a', 'b', 'c', 'd']
for i, ax in enumerate(axs.flat):
    ax.text(-0.1, 1.15, labels[i], transform=ax.transAxes, fontsize=20, fontweight='bold', va='top', ha='right')


# Adjust layout
plt.tight_layout()
# fig.suptitle("Encoder Performance vs. Normalized Layer Depth", fontsize=25, y=1.10)
# Add common x-label and y-label
fig.text(0.5, -0.01, 'Normalized Layer Depth', ha='center', fontsize=20)
fig.text(-0.03, 0.5, 'Encoder Performance (Avg $r^2$)', va='center', rotation='vertical', fontsize=20)

#plt.savefig('./FIGURES/scaling_figure.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Define Groups for Cleaner Storytelling
llama_models = [
    "LLAMA7B", "LLAMA13B", "LLAMA30B", "LLAMA65B"
]

pythia_models = [
    "EleutherAI/pythia-14m", "EleutherAI/pythia-70m", "EleutherAI/pythia-160m"
]

# Grouping the BabyLM models by size track (10M vs 100M) allows for fair comparison
babylm_10m = [
    "babylm-babyllama-10m-2024",
    "babylm/opt-125m-strict-small-2023",
    "babylm/ltgbert-10m-2024",
    "babylm/t5-base-strict-small-2023"
]

babylm_100m = [
    "babylm/babyllama-100m-2024",
    "babylm/opt-125m-strict-2023",
    "babylm/ltgbert-100m-2024",
    "babylm/t5-base-strict-2023"
]

# 2. Define a clean color/marker map
# Using distinct colors/markers helps distinguish lines even in black & white
STYLES = {
    # LLaMA Family - Blues
    "LLAMA7B": {"color": "#a1dab4", "marker": "o"},
    "LLAMA13B": {"color": "#41b6c4", "marker": "^"},
    "LLAMA30B": {"color": "#2c7fb8", "marker": "s"},
    "LLAMA65B": {"color": "#253494", "marker": "D"},
    
    # Pythia - Reds
    "EleutherAI/pythia-14m": {"color": "#fecc5c", "marker": "o"},
    "EleutherAI/pythia-70m": {"color": "#fd8d3c", "marker": "^"},
    "EleutherAI/pythia-160m": {"color": "#f03b20", "marker": "s"},

    # Architecture Comparisons (Consistent colors across 10M and 100M plots)
    # BabyLLaMA (Purple)
    "babylm-babyllama-10m-2024": {"color": "tab:purple", "marker": "o"},
    "babylm/babyllama-100m-2024": {"color": "tab:purple", "marker": "o"},
    # OPT (Gray)
    "babylm/opt-125m-strict-small-2023": {"color": "tab:gray", "marker": "^"},
    "babylm/opt-125m-strict-2023": {"color": "tab:gray", "marker": "^"},
    # LTGBERT (Blue)
    "babylm/ltgbert-10m-2024": {"color": "tab:blue", "marker": "s"},
    "babylm/ltgbert-100m-2024": {"color": "tab:blue", "marker": "s"},
    # T5 (Green)
    "babylm/t5-base-strict-small-2023": {"color": "tab:green", "marker": "D"},
    "babylm/t5-base-strict-2023": {"color": "tab:green", "marker": "D"},
}

# 3. The Plotting Function
def plot_subplot(ax, models, title, y_limit=None):
    # Filter dataset
    df_base = results['encoding_setting__response_setting'][("default", "default")]
    
    for model in models:
        # Identify base key for coloring
        model_key = model.split('_')[0]
        style = STYLES.get(model_key, {"color": "black", "marker": "."})
        
        # Extract data for this model
        # Note: We escape regex characters to be safe
        df_model = df_base[df_base['stimulus_setting'].str.contains(r'^' + model, na=False)]
        
        # Process using your existing utility function
        numerized_df = filter_sort_and_convert_to_numeric(
            df_model, {}, multimodal=False, sort_by=x_axis, colunmns2convert=['stimulus_setting']
        )
        
        # Normalize layers (0 to 1)
        layers = numerized_df[x_axis] / (NUM_LAYERS_DICT[model_key] - 1)
        scores = numerized_df[criteria]
        
        ax.plot(
            layers, 
            scores, 
            label=model2name_dict.get(model_key, model_key), 
            color=style["color"],
            marker=style["marker"],
            linewidth=2,
            markersize=6,
            alpha=0.8
        )

    # Aesthetics
    ax.set_title(title, fontsize=14, pad=10)
    ax.set_xlim(0, 1)
    if y_limit:
        ax.set_ylim(y_limit)
    
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.tick_params(axis='both', which='major', labelsize=10)
    
    # Legend outside the plot
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., fontsize=10)

# 4. Generate the Figure
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Plot A: LLaMA Scaling
plot_subplot(axs[0, 0], llama_models, "(a) LLaMA Parameter Scaling")

# Plot B: Pythia Scaling
plot_subplot(axs[0, 1], pythia_models, "(b) Pythia Parameter Scaling")

# Plot C: BabyLM 10M Track (Use tight Y-limits here if scores are low!)
plot_subplot(axs[1, 0], babylm_10m, "(c) BabyLM 10M Track Comparison")

# Plot D: BabyLM 100M Track
plot_subplot(axs[1, 1], babylm_100m, "(d) BabyLM 100M Track Comparison")

# Common Labels
fig.text(0.5, 0.02, 'Normalized Layer Depth', ha='center', fontsize=16)
fig.text(0.02, 0.5, 'Encoder Performance (Avg $r^2$)', va='center', rotation='vertical', fontsize=16)

plt.tight_layout(rect=[0.03, 0.03, 1, 1]) # Make room for common labels
plt.show()

In [ ]:
# from utils_perf_df import analyze_custom_settings_combinations, filter_sort_and_convert_to_numeric
# from utils_plotter import plot_baseline_value # Assuming this is correctly imported
# import matplotlib.pyplot as plt
# import pandas as pd
# import numpy as np 

# # Assume NUM_LAYERS_DICT is defined somewhere, adding a placeholder for demonstration
# # You should ensure your actual NUM_LAYERS_DICT is defined before this line
# NUM_LAYERS_DICT = {
#     "LLAMA7B": 32,
#     "LLAMA13B": 40,
#     "LLAMA30B": 60,
#     "LLAMA65B": 80,
#     "LLAMA27Bhf": 32,
#     "LLAMA38B": 32,
#     "babylm-babyllama-10m-2024": 12 
# }


# all_results = pd.read_csv("./all_results_old_and_new.csv")

# columns_of_interest = [ 'mean_cc_norm_mean', 'mean_cc_norm_std', 'mean_r2_mean', 'mean_r2_std', 'stimulus_setting', 'stimulus_setting_2', 'encoding_setting', 'response_setting', 'format_name_name']

# final_results = all_results[columns_of_interest]
# final_results = final_results.reset_index(drop = True)
# final_results = final_results[final_results['format_name_name'].str.contains('loss_criteria__MAE|default')]

# settings_list = ['stimulus_setting', 'encoding_setting', 'response_setting', 'stimulus_setting_2']
# results = analyze_custom_settings_combinations(final_results, settings_list)



# # Define the model to name dictionary
# model2name_dict = {
#     "LLAMA7B": "LLAMA 7B",
#     "LLAMA13B": "LLAMA 13B",
#     "LLAMA30B": "LLAMA 33B",
#     "LLAMA65B": "LLAMA 65B",
#     "LLAMA27Bhf": "LLAMA-2 7B",
#     "LLAMA38B": "LLAMA-3 8B",
#     "babylm-babyllama-10m-2024": "BabyLLaMA 58M 10M"
# }

# # Define the criteria and x-axis
# criteria = 'mean_r2_mean'
# x_axis = 'stimulus_setting'

# # 1. MODIFIED: Function to apply extra plot settings (NO LEGEND/BASELINE PLOTTING HERE)
# def extra_plot_settings_main(ax, title, xlim, ylim, criteria, xlabel="Normalized Layer Depth", ylabel="Encoder Performance (Avg $r^2$)", fontsize=16, **kwargs):
#     # Only set labels if they are NOT None (allowing global fig.text to override)
#     if xlabel is not None:
#         ax.set_xlabel(xlabel, fontsize=fontsize)
#     if ylabel is not None:
#         ax.set_ylabel(ylabel, fontsize=fontsize)
    
#     ax.set_title(title, fontsize=fontsize)
#     ax.set_xlim(xlim)
#     ax.set_ylim(ylim)
#     ax.tick_params(axis='both', labelsize=fontsize)
#     # ax.legend(...) # Removed
#     # plot_baseline_value(...) # Removed


# # 2. MODIFIED: Function to plot data and return legend handles/labels
# def plot_models_and_return_legend(ax, models, title, xlabel, ylabel, xlim, ylim, fontsize = 12, loc = 'lower right', **kwargs):
#     for model in models:
#         df = results['encoding_setting__response_setting'][("default", "default")]
#         # Using a raw string r'' for the regex pattern
#         df = df[df['stimulus_setting'].str.contains(r'^' + model, na=False)]
#         numerized_df = filter_sort_and_convert_to_numeric(df, {}, multimodal=False, sort_by=x_axis,
#                                                           colunmns2convert=['stimulus_setting'])
#         ax.plot(numerized_df[x_axis] / (NUM_LAYERS_DICT[model.split('_')[0]] - 1), numerized_df[criteria], label=model2name_dict[model], marker='+', linewidth=3)
    
#     # Run extra settings for the main plot
#     # The default labels are used if xlabel/ylabel are None
#     extra_plot_settings_main(ax, title, xlim, ylim, criteria, xlabel, ylabel, fontsize = fontsize)
    
#     # Run the baseline plotting to ensure the line is on the main figure
#     plot_baseline_value(criteria, ax_or_plt=ax, **kwargs)

#     # Collect all handles/labels (models + baseline) for the external legend
#     all_handles, all_labels = ax.get_legend_handles_labels()
    
#     return all_handles, all_labels, loc


# # Create a 2x2 grid of subplots for the main figure
# fig, axs = plt.subplots(2, 2, figsize=(10, 7))

# # A list to store legend data for each subplot
# legend_data = [] # Stores: [(handles, labels, loc), ...]

# # --- START OF CORRECTED CALLS ---

# # Plot A: LLAMA 7B Generations
# handles_a, labels_a, loc_a = plot_models_and_return_legend(
#     axs[0, 0], 
#     ["LLAMA7B", "LLAMA27Bhf", "LLAMA38B"], 
#     title="(a) LLAMA 7B Generations", 
#     xlabel=None, 
#     ylabel=None, 
#     xlim=(0, 1), 
#     ylim=(0, 0.04)
# )
# legend_data.append((handles_a, labels_a, loc_a))

# # Plot B: LLAMA Parameter Scaling
# handles_b, labels_b, loc_b = plot_models_and_return_legend(
#     axs[0, 1], 
#     ["LLAMA7B", "LLAMA13B", "LLAMA30B", "LLAMA65B"], 
#     title="(b) LLAMA Parameter Scaling", 
#     xlabel=None, 
#     ylabel=None, 
#     xlim=(0, 1), 
#     ylim=(0, 0.04)
# )
# legend_data.append((handles_b, labels_b, loc_b))

# # Plot C: BabyLLaMA
# handles_c, labels_c, loc_c = plot_models_and_return_legend(
#     axs[1, 0], 
#     ["babylm-babyllama-10m-2024"], 
#     title="(c) BabyLLaMA", 
#     xlabel=None, 
#     ylabel=None, 
#     xlim=(0, 1), 
#     ylim=(0, 0.004)
# )
# legend_data.append((handles_c, labels_c, loc_c))

# # Plot D: All Models Comparison
# handles_d, labels_d, loc_d = plot_models_and_return_legend(
#     axs[1, 1], 
#     ["LLAMA7B", "LLAMA13B", "LLAMA30B", "LLAMA65B", "babylm-babyllama-10m-2024"], 
#     title="(d) All Models Comparison", 
#     xlabel=None, 
#     ylabel=None, 
#     xlim=(0, 1), 
#     ylim=(0, 0.04), 
#     loc ='center right'
# )
# legend_data.append((handles_d, labels_d, loc_d))

# # --- END OF CORRECTED CALLS ---

# # Adjust layout
# plt.tight_layout()
# # Add common x-label and y-label using fig.text
# fig.text(0.5, -0.01, 'Normalized Layer Depth', ha='center', fontsize=20)
# fig.text(-0.03, 0.5, 'Encoder Performance (Avg $r^2$)', va='center', rotation='vertical', fontsize=20)

# # Save the main figure WITHOUT any legend
# plt.savefig('./FIGURES/scaling_figure_no_legend.png', dpi=300, bbox_inches='tight')
# plt.close(fig) # Close the main figure


# # 5. Function to save the four legends separately
# def save_legend_separately(handles, labels, filename, fontsize=12, ncol=1):
#     # Create a new, temporary figure just for the legend
#     legend_fig = plt.figure(figsize=(10, 1)) 
#     legend_fig.patch.set_alpha(0.0) # Make figure background transparent
    
#     # Create the legend on this figure
#     legend = legend_fig.legend(handles, labels, loc='center', ncol=ncol, frameon=False, fontsize=fontsize)
    
#     # Save the legend by calculating the exact bounding box
#     renderer = legend_fig.canvas.get_renderer()
#     extent = legend.get_window_extent(renderer=renderer).transformed(legend_fig.dpi_scale_trans.inverted())
    
#     legend_fig.savefig(filename, dpi=300, bbox_inches=extent, pad_inches=0.1, transparent=True)
#     plt.close(legend_fig) # Close the temporary figure


# # Save each legend
# for i, (handles, labels, loc) in enumerate(legend_data):
#     # Use a larger ncol for the 'All Models Comparison' for better formatting
#     ncol = 2 if i == 3 else 1 
#     save_legend_separately(handles, labels, f'./FIGURES/scaling_figure_legend_{i+1}.png', fontsize=12, ncol=ncol)

# # plt.show()